In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## sklearn
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import randint

# metrics
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error, mean_squared_error, r2_score

##xgboost
import xgboost as xgb

#Guardar modelo
from joblib import dump
import joblib

sns.set(style="whitegrid")  # estilo bonito para seaborn

# Explicabilidad
import shap



     


In [ ]:
# Cargar y explorar el dataset
dataset = load_dataset("rajistics/electricity_demand", split="train")
df = pd.DataFrame(dataset)
print("Primeras filas del dataset:")
print(df.head())


In [ ]:
# Preprocesamiento
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp')
df['date'] = df['timestamp'].dt.date

In [ ]:
# Agrupar por día y asegurar 48 intervalos por día
daily_series = df.groupby('date')['demand'].apply(list)
daily_series = daily_series[daily_series.apply(lambda x: len(x) == 48)]

In [ ]:
# Crear dataset supervisado (día N -> día N+1)
X = []
y = []
for i in range(len(daily_series) - 1):
    X.append(daily_series.iloc[i])
    y.append(daily_series.iloc[i + 1])

X = np.array(X)
y = np.array(y)

In [ ]:
# Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)


In [ ]:
# Entrenar modelo Random Forest
# model = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, random_state=42))
# model.fit(X_train, y_train)

# Modelos a evaluar
models = {
    "LinearRegressionH": LinearRegression(fit_intercept=True),
    
    "DecisionTreeRegressorH": DecisionTreeRegressor(
        max_depth=20, 
        min_samples_split=5, 
        min_samples_leaf=2, 
        max_features='sqrt',
        random_state=42
    ),
    
    "RandomForestRegressorH": RandomForestRegressor(
        n_estimators=500, 
        max_depth=30,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        random_state=42
    ),
    
    "XGBRegressorH": xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1,
        random_state=42,
        tree_method="hist"  # Aceleración para CPU
    )
}

In [ ]:
# Diccionario para almacenar resultados
results = {
    "Model": [],
    "MAE": [],
    "MSE": [],
    "RMSE": [],
    "R2": []
}


In [ ]:
for name, model in models.items():
    print("################################################")
    print(f"########################  {name}")
    print("################################################\n")

    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Calcular métricas
        MAE = mean_absolute_error(y_test, y_pred)
        MSE = mean_squared_error(y_test, y_pred)
        RMSE = np.sqrt(MSE)
        R2 = r2_score(y_test, y_pred)

        # Guardar resultados
        results["Model"].append(name)
        results["MAE"].append(MAE)
        results["MSE"].append(MSE)
        results["RMSE"].append(RMSE)
        results["R2"].append(R2)

        print(f"MAE: {MAE:.2f}, MSE: {MSE:.2f}, RMSE: {RMSE:.2f}, R2: {R2:.2f}")

        # Guardar modelo entrenado
        dump(model, f"modelos-forecasting/{name}.joblib")
        print(f"Modelo {name} guardado correctamente.\n")

        # -----------------------------------
        # 1. Graficar Real vs Predicho
        plt.figure(figsize=(8,6))
        plt.scatter(y_test, y_pred, alpha=0.7)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
        plt.xlabel('Ventas Reales')
        plt.ylabel('Ventas Predichas')
        plt.title(f'Real vs Predicho - {name}')
        plt.grid()
        plt.show()

        # -----------------------------------
        # 2. Graficar histograma de residuos
        residuals = y_test - y_pred
        plt.figure(figsize=(8,5))
        sns.histplot(residuals, bins=30, kde=True)
        plt.title(f'Histograma de Errores (Residuos) - {name}')
        plt.xlabel('Error')
        plt.ylabel('Frecuencia')
        plt.grid()
        plt.show()

    except Exception as e:
        print(f"Error en modelo {name}: {e}")

In [ ]:
# Mostrar resumen de resultados
results_df = pd.DataFrame(results)
results_df.sort_values(by="R2", ascending=False)


In [ ]:
# Comparación de R2
plt.figure(figsize=(10,6))
sns.barplot(x="Model", y="R2", data=results_df)
plt.title("Comparación del Coeficiente R2 entre Modelos")
plt.xlabel("Modelo")
plt.ylabel("R2 Score")
plt.ylim(0,1)
plt.show()

# Explicabiidad global y local

## Local

In [ ]:

model = joblib.load("modelos-forecasting/RandomForestRegressorH.joblib")


explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

np.shape(shap_values.values)

In [ ]:
shap.plots.waterfall(shap_values[0])

## Global

In [ ]:


# Calcular la importancia global promediando los valores absolutos
shap_importance = np.abs(shap_values.values).mean(axis=0)

# Crear DataFrame con la importancia global
importance_df = pd.DataFrame({'Feature': X_train.columns.tolist(), 'SHAP Importance': shap_importance})
importance_df = importance_df.sort_values(by="SHAP Importance", ascending=False)

importance_df

In [ ]:
# Visualización de importancia global de características
shap.summary_plot(shap_values, X_train)


In [ ]:
### Absolute MEAN SHAP

shap.plots.bar(shap_values)